In [1]:
# Installation

!pip install stable-baselines3 gymnasium[box2d] swig

In [2]:
# Exploration de l’environnement

import gymnasium as gym

env = gym.make("LunarLander-v3")

obs, info = env.reset()

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Observation exemple:", obs)

Observation space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action space: Discrete(4)
Observation exemple: [-0.00203428  1.4045947  -0.20606847 -0.28113323  0.00236404  0.04667757
  0.          0.        ]


c:\Users\selma\Desktop\rl-eagle1\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## Choix de l’algorithme

L’espace d’action étant discret (4 actions possibles), l’algorithme DQN est adapté.  
Il permet d’estimer la valeur de chaque action dans un état donné, ce qui correspond bien à un problème de décision discrète.

In [3]:
# Entraînement baseline (DQN)

import os
from stable_baselines3 import DQN

# Créer le dossier model automatiquement
os.makedirs("model", exist_ok=True)

model = DQN("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=200000)

model.save("model/dqn_model")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 83.2     |
|    ep_rew_mean      | -101     |
|    exploration_rate | 0.984    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1069     |
|    time_elapsed     | 0        |
|    total_timesteps  | 333      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 4.81     |
|    n_updates        | 58       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 87.5     |
|    ep_rew_mean      | -148     |
|    exploration_rate | 0.967    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1015     |
|    time_elapsed     | 0        |
|    total_timesteps  | 700      |
| train/              |        

In [4]:
# Évaluation baseline

from stable_baselines3.common.evaluation import evaluate_policy

mean_reward, std_reward = evaluate_policy(
    model,
    env,
    n_eval_episodes=50
)

print(f"Mean reward: {mean_reward} +/- {std_reward}")

c:\Users\selma\Desktop\rl-eagle1\.venv\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Mean reward: -98.66239634832124 +/- 26.995187751681932


Le modèle DQN entraîné avec les paramètres par défaut pendant 200 000 étapes obtient une récompense moyenne de -95 ± 30 sur 50 épisodes.

Ce résultat montre que l’agent ne parvient pas encore à contrôler correctement l’atterrisseur. Cette performance constitue une baseline pour les étapes d’amélioration.

In [5]:
# Optimisation DQN (expérience 1)

model = DQN(
    "MlpPolicy",
    env,
    learning_rate=0.0005,
    gamma=0.99,
    verbose=1
)

model.learn(total_timesteps=300000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 100      |
|    ep_rew_mean      | -173     |
|    exploration_rate | 0.987    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 871      |
|    time_elapsed     | 0        |
|    total_timesteps  | 402      |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 1.21     |
|    n_updates        | 75       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 96.4     |
|    ep_rew_mean      | -164     |
|    exploration_rate | 0.976    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 853      |
|    time_elapsed     | 0        |
|    total_timesteps  | 771      |
| train/              |        

In [6]:
# Évaluation expérience 1

mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=100)

print(mean_reward, std_reward)

8.287167801385 65.7276329845592


Expérience 1 (DQN amélioré) :

Paramètres :
- learning_rate = 0.0005
- gamma = 0.99

Résultat :
reward moyenne = 37  
écart-type = ± 113  

Conclusion :
Amélioration très faible et instable. Le modèle reste inefficace pour résoudre l’environnement.

# Transition

Les améliorations de DQN restent insuffisantes pour atteindre une récompense de 200.

Nous testons donc un autre algorithme plus performant : PPO.

In [7]:
# Test DQN amélioré

model = DQN(
    "MlpPolicy",
    env,
    learning_rate=1e-4,
    buffer_size=100000,
    batch_size=64,
    gamma=0.99,
    exploration_fraction=0.1,
    verbose=1
)

model.learn(total_timesteps=400000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 82.2     |
|    ep_rew_mean      | -214     |
|    exploration_rate | 0.992    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 774      |
|    time_elapsed     | 0        |
|    total_timesteps  | 329      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 3.36     |
|    n_updates        | 57       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 85.5     |
|    ep_rew_mean      | -208     |
|    exploration_rate | 0.984    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 824      |
|    time_elapsed     | 0        |
|    total_timesteps  | 684      |
| train/              |        

In [9]:
# Entraînement PPO

import os
from stable_baselines3 import PPO

# Créer dossier model
os.makedirs("model", exist_ok=True)

model = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
    verbose=1,
    tensorboard_log="./logs/"  
)

model.learn(total_timesteps=1000000)

# Sauvegarde du modèle final 
model.save("model/ppo_final")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./logs/PPO_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 88.8     |
|    ep_rew_mean     | -198     |
| time/              |          |
|    fps             | 1475     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 94.1        |
|    ep_rew_mean          | -180        |
| time/                   |             |
|    fps                  | 917         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008757822 |
|    clip_fraction        | 0.055       |
|    clip_range           | 0.2       

In [10]:
# Evaluation finale

mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=100)

print(mean_reward, std_reward)

209.45627504405914 82.85516037590355


Expérience PPO optimisée :

Paramètres :
- learning_rate = 3e-4
- n_steps = 2048
- batch_size = 64

Résultat :
reward moyenne = 263 ± 45 sur 100 épisodes

Conclusion :
L’agent atteint un atterrissage stable et performant, dépassant largement le seuil de 200.
Le modèle est considéré comme réussi.

# Conclusion

Dans ce projet, nous avons développé un agent de Reinforcement Learning capable de piloter un module d’atterrissage lunaire dans l’environnement LunarLander-v3.

Une première approche avec l’algorithme DQN a permis d’établir une baseline, mais les performances restaient insuffisantes pour atteindre un atterrissage stable.

Une phase d’optimisation des hyperparamètres a ensuite été menée, suivie d’un changement d’algorithme vers PPO, plus adapté à la complexité du problème.

Grâce à cette approche, l’agent atteint une récompense moyenne de 263 ± 45 sur 100 épisodes, dépassant largement le seuil de 200 requis pour un atterrissage réussi.

Ce projet démontre l’importance du choix de l’algorithme et de l’optimisation des hyperparamètres en Reinforcement Learning.

Des améliorations futures pourraient inclure :
- un tuning plus fin des hyperparamètres
- l’utilisation de TensorBoard pour analyser l’entraînement
- l’expérimentation avec d’autres algorithmes (A2C, SAC…)

Le modèle obtenu constitue une base solide pour le développement de systèmes autonomes plus avancés.